# Pipeline consolidado: discretização simples, discretização trigonométrica e janelamento

Ajuste `CONFIG["csv_path"]` para o caminho do CSV completo, incluindo os dados de 2016 quando disponíveis.

In [1]:
from __future__ import annotations

import json
import random
import warnings
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

import joblib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.models import Sequential, load_model

warnings.filterwarnings("ignore")

I0000 00:00:1779720563.108136  198991 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
CONFIG: Dict[str, Any] = {
    "csv_path": "../../anemo-mucuri/anemo-mucuri-150m-dez2015.csv",
    "target_col": "v_anemo2",
    "output_dir": "resultados_vento_64_32_sem_dropout",
    "max_janela": 10,
    "epochs": 200,
    "batch_size": 8,
    "patience": 25,
    "learning_rate": 0.001,
    "data_inicio_ref": "2015-11-30 14:00:00",
    "data_fim_ref": "2015-12-23 11:00:00",
    "data_inicio_prev": "2015-12-23 12:00:00",
    "data_fim_prev": "2015-12-31 13:00:00",
    "train_size": 0.70,
    "val_size": 0.15,
    "test_size": 0.15,
    "comparar_por_bloco": "previsao", # simulacao_artigo, teste ou futuro_2016
    "comparar_por_metrica": "mse",    # menor é melhor: mae, mse, rmse, erro_maximo
    "random_seed": 42,
}

In [3]:
def configurar_semente(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def criar_pastas(output_dir: str, nome_experimento: str) -> dict[str, Path]:
    base_dir = Path(output_dir) / nome_experimento
    paths = {
        "base": base_dir,
        "modelos": base_dir / "modelos",
        "scalers": base_dir / "scalers",
        "metricas": base_dir / "metricas",
        "historicos": base_dir / "historicos",
        "previsoes": base_dir / "previsoes",
        "graficos": base_dir / "graficos",
        "json": base_dir / "json",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def salvar_json(objeto: Any, caminho: Path) -> None:
    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(objeto, arquivo, indent=4, ensure_ascii=False)


def carregar_dataset(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df[df['ano'] == 2015].reset_index(drop=True)
    colunas_temporais = ["ano", "mes", "dia", "hora"]
    colunas_ausentes = [col for col in colunas_temporais if col not in df.columns]
    if colunas_ausentes:
        raise ValueError(f"Colunas temporais ausentes no CSV: {colunas_ausentes}")

    df = df.copy()
    for coluna in df.columns:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    df = df.dropna(subset=colunas_temporais).copy()
    df["datahora"] = pd.to_datetime(
        df[colunas_temporais].rename(
            columns={"ano": "year", "mes": "month", "dia": "day", "hora": "hour"}
        ),
        errors="coerce",
    )
    df = df.dropna(subset=["datahora"]).sort_values("datahora").reset_index(drop=True)
    print("Dataset")
    display(df)
    print()
    return df


def validar_dataset(df: pd.DataFrame, target_col: str) -> None:
    if target_col not in df.columns:
        raise ValueError(f"Coluna alvo '{target_col}' não encontrada no dataset.")
    if df[target_col].dropna().empty:
        raise ValueError(f"Coluna alvo '{target_col}' não possui valores válidos.")

In [ ]:
def adicionar_seno_cosseno(
    df: pd.DataFrame, coluna: str, periodo: float, prefixo: Optional[str] = None
) -> pd.DataFrame:
    prefixo = prefixo or coluna
    angulo = 2.0 * np.pi * df[coluna].astype(float) / periodo
    df[f"{prefixo}_sin"] = np.sin(angulo)
    df[f"{prefixo}_cos"] = np.cos(angulo)
    return df


def criar_dataset_discretizacao_simples(
    df: pd.DataFrame, target_col: str
) -> pd.DataFrame:
    """Usa as variáveis atuais para prever a velocidade na próxima hora."""
    df_modelo = df.copy()
    df_modelo["y"] = df_modelo[target_col].shift(-1)
    df_modelo["datahora_y"] = df_modelo["datahora"].shift(-1)
    return df_modelo.dropna().reset_index(drop=True)


def criar_dataset_discretizacao_trigonometrica(
    df: pd.DataFrame, target_col: str
) -> pd.DataFrame:
    """Representa variáveis cíclicas por seno/cosseno e prevê a próxima hora."""
    df_modelo = df.copy()
    df_modelo = adicionar_seno_cosseno(df_modelo, "mes", 12, "mes")
    df_modelo = adicionar_seno_cosseno(df_modelo, "hora", 24, "hora")
    df_modelo = adicionar_seno_cosseno(df_modelo, "dir_1", 360, "dir_1")

    df_modelo = df_modelo.drop(columns=["mes", "hora", "dir_1"], errors="ignore")
    df_modelo["y"] = df_modelo[target_col].shift(-1)
    df_modelo["datahora_y"] = df_modelo["datahora"].shift(-1)

    print()
    return df_modelo.dropna().reset_index(drop=True)


def criar_dataset_janelamento(
    df: pd.DataFrame, target_col: str, janela: int
) -> pd.DataFrame:
    """Cria amostras com os últimos `janela` passos para prever t+1."""
    if janela < 1:
        raise ValueError("A janela deve ser maior ou igual a 1.")

    base = df.copy()
    feature_cols = [col for col in base.columns if col != "datahora"]
    registros: list[dict[str, Any]] = []

    for fim_janela in range(janela - 1, len(base) - 1):
        inicio_janela = fim_janela - janela + 1
        linha: dict[str, Any] = {
            "datahora": base.loc[fim_janela, "datahora"],
            "datahora_y": base.loc[fim_janela + 1, "datahora"],
            "y": base.loc[fim_janela + 1, target_col],
        }
        janela_df = base.loc[inicio_janela:fim_janela, feature_cols]
        for atraso, (_, row) in enumerate(janela_df.iterrows(), start=janela - 1):
            # atraso maior = observação mais antiga; lag_0 = observação imediatamente anterior ao target.
            lag = janela - atraso - 1
            for coluna in feature_cols:
                linha[f"{coluna}_lag_{lag}"] = row[coluna]
        registros.append(linha)
    return pd.DataFrame(registros).dropna().reset_index(drop=True)


def selecionar_features(df_modelo: pd.DataFrame) -> list[str]:
    ignorar = {"y", "datahora", "datahora_y"}
    return [col for col in df_modelo.columns if col not in ignorar]


def dividir_temporalmente(df_modelo: pd.DataFrame, config: dict[str, Any]) -> dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """Divide em treino/val/teste e blocos de simulação usando a data do alvo previsto."""
    feature_cols = selecionar_features(df_modelo)
    data_col = "datahora_y" if "datahora_y" in df_modelo.columns else "datahora"

    dt_ini_ref = pd.to_datetime(config["data_inicio_ref"])
    dt_fim_ref = pd.to_datetime(config["data_fim_ref"])
    dt_ini_sim = pd.to_datetime(config["data_inicio_prev"])
    dt_fim_sim = pd.to_datetime(config["data_fim_prev"])

    df_ref = df_modelo[(df_modelo[data_col] >= dt_ini_ref) & (df_modelo[data_col] <= dt_fim_ref)].copy()
    df_sim = df_modelo[(df_modelo[data_col] >= dt_ini_sim) & (df_modelo[data_col] <= dt_fim_sim)].copy()

    if df_ref.empty:
        raise ValueError("O período de referência ficou vazio. Verifique as datas em CONFIG.")

    n = len(df_ref)
    train_end = int(n * float(config["train_size"]))
    val_end = int(n * (float(config["train_size"]) + float(config["val_size"])))
    if train_end <= 0 or val_end <= train_end or val_end >= n:
        raise ValueError(f"Divisão temporal inválida para {n} amostras de referência.")

    blocos_df = {
        "treino": df_ref.iloc[:train_end],
        "validacao": df_ref.iloc[train_end:val_end],
        "teste": df_ref.iloc[val_end:],
        "previsao": df_sim,
    }

    blocos: dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}
    for nome, bloco in blocos_df.items():
        if bloco.empty:
            blocos[nome] = (np.empty((0, len(feature_cols))), np.array([]), np.array([]))
            continue
        X = bloco[feature_cols].astype(float).to_numpy()
        y = bloco["y"].astype(float).to_numpy()
        datas = bloco[data_col].to_numpy()
        blocos[nome] = (X, y, datas)
    return blocos


def criar_modelo_mlp(input_dim: int, learning_rate: float) -> Sequential:
    model = Sequential(
        [
            Input(shape=(input_dim,)),
            Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
            Dense(32, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.001)),
            Dense(1, activation="linear"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mean_absolute_error", "mean_squared_error"],
    )
    return model


def calcular_metricas(y_real: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_real = np.asarray(y_real).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    if len(y_real) == 0:
        return {
            "mae": np.nan,
            "mse": np.nan,
            "rmse": np.nan,
            "erro_minimo": np.nan,
            "erro_maximo": np.nan,
            "r2": np.nan,
            "pearson_r": np.nan,
        }

    erros_abs = np.abs(y_real - y_pred)
    mse = mean_squared_error(y_real, y_pred)
    pearson_r = pearsonr(y_real, y_pred)[0] if len(y_real) > 1 else np.nan
    return {
        "mae": float(mean_absolute_error(y_real, y_pred)),
        "mse": float(mse),
        "rmse": float(np.sqrt(mse)),
        "erro_minimo": float(np.min(erros_abs)),
        "erro_maximo": float(np.max(erros_abs)),
        "r2": float(r2_score(y_real, y_pred)) if len(y_real) > 1 else np.nan,
        "pearson_r": float(pearson_r),
    }


def salvar_grafico_loss(
    history: tf.keras.callbacks.History, paths: dict[str, Path]
) -> None:
    plt.figure(figsize=(12, 5))
    plt.plot(history.history["loss"], label="Treino")
    plt.plot(history.history["val_loss"], label="Validação")
    plt.title("Loss durante o treinamento")
    plt.xlabel("Época")
    plt.ylabel("MSE")
    plt.legend()
    # plt.grid(True)
    plt.tight_layout()
    plt.savefig(paths["graficos"] / "loss.png", dpi=300)
    plt.close()


def salvar_grafico_previsao(
    datas: Iterable[Any],
    y_real: np.ndarray,
    y_pred: np.ndarray,
    paths: dict[str, Path],
    sufixo: str,
) -> None:
    if len(y_real) == 0:
        return
    plt.figure(figsize=(14, 5))
    plt.plot(datas, y_real, label="Real", linestyle="--")
    plt.plot(datas, y_pred, label="Previsto")
    plt.title(f"Velocidade real vs prevista - {sufixo}")
    plt.xlabel("Data e hora")
    plt.ylabel("Velocidade do vento - v_anemo2 (m/s)")
    plt.legend()
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y %H:%M"))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(paths["graficos"] / f"real_vs_previsto_{sufixo}.png", dpi=300)
    plt.close()


def salvar_grafico_residuos(
    datas: Iterable[Any],
    y_real: np.ndarray,
    y_pred: np.ndarray,
    paths: dict[str, Path],
    sufixo: str,
) -> None:
    if len(y_real) == 0:
        return
    residuos = np.asarray(y_real).reshape(-1) - np.asarray(y_pred).reshape(-1)
    plt.figure(figsize=(14, 5))
    plt.plot(datas, residuos)
    plt.axhline(0, linestyle="--")
    plt.title(f"Resíduos - {sufixo}")
    plt.xlabel("Data e hora")
    plt.ylabel("Erro real - previsto (m/s)")
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y %H:%M"))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(paths["graficos"] / f"residuos_{sufixo}.png", dpi=300)
    plt.close()


def exportar_previsoes(
    nome_bloco: str,
    datas: np.ndarray,
    y_real: np.ndarray,
    y_pred: np.ndarray,
    paths: dict[str, Path],
) -> dict[str, float]:
    metricas = calcular_metricas(y_real, y_pred)
    metricas["n_amostras"] = int(len(y_real))
    salvar_json(metricas, paths["metricas"] / f"metricas_{nome_bloco}.json")

    if len(y_real) > 0:
        df_prev = pd.DataFrame(
            {
                "datahora": pd.to_datetime(datas),
                "y_real": y_real.reshape(-1),
                "y_pred": y_pred.reshape(-1),
                "residuo": y_real.reshape(-1) - y_pred.reshape(-1),
            }
        )
        df_prev.to_csv(paths["previsoes"] / f"previsoes_{nome_bloco}.csv", index=False)
        salvar_grafico_previsao(
            df_prev["datahora"],
            df_prev["y_real"].to_numpy(),
            df_prev["y_pred"].to_numpy(),
            paths,
            nome_bloco,
        )
        salvar_grafico_residuos(
            df_prev["datahora"],
            df_prev["y_real"].to_numpy(),
            df_prev["y_pred"].to_numpy(),
            paths,
            nome_bloco,
        )
    return metricas


def treinar_e_avaliar(
    df_modelo: pd.DataFrame, config: dict[str, Any], nome_experimento: str
) -> dict[str, Any]:
    paths = criar_pastas(config["output_dir"], nome_experimento)
    df_modelo.to_csv(paths["base"] / "dataset_transformado.csv", index=False)

    blocos = dividir_temporalmente(df_modelo, config)
    X_train, y_train, _ = blocos["treino"]
    X_val, y_val, _ = blocos["validacao"]

    print("\nBlocos criados:")
    for nome, (X, y, datas) in blocos.items():
        print(
            f"- {nome}: {len(X)} amostras | período: {pd.to_datetime(datas).min()} até {pd.to_datetime(datas).max()}"
        )

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = criar_modelo_mlp(
        input_dim=X_train_scaled.shape[1], learning_rate=float(config["learning_rate"])
    )
    tf.keras.utils.plot_model(
        model,
        to_file=paths["graficos"] / "arquitetura_mlp.png",
        show_shapes=True,
        show_layer_names=True,
        dpi=300
    )
    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=int(config["patience"]),
            restore_best_weights=True,
        ),
        ModelCheckpoint(
            paths["modelos"] / "melhor_modelo.keras",
            monitor="val_loss",
            save_best_only=True,
        ),
    ]

    history = model.fit(
        X_train_scaled,
        y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=int(config["epochs"]),
        batch_size=int(config["batch_size"]),
        verbose=0,
        callbacks=callbacks,
    )

    best_model_path = paths["modelos"] / "melhor_modelo.keras"
    if best_model_path.exists():
        model = load_model(best_model_path)
    model.save(paths["modelos"] / "modelo_final.keras")
    joblib.dump(scaler, paths["scalers"] / "scaler.pkl")
    salvar_json(config, paths["json"] / "config.json")
    salvar_json(
        {k: [float(v) for v in vals] for k, vals in history.history.items()},
        paths["historicos"] / "historico.json",
    )
    salvar_grafico_loss(history, paths)

    metricas_por_bloco: dict[str, dict[str, float]] = {}
    previsoes_por_bloco: dict[str, pd.DataFrame] = {}

    for nome_bloco in ["treino", "validacao", "teste", "previsao"]:
        X, y, datas = blocos[nome_bloco]
        if len(y) == 0:
            metricas_por_bloco[nome_bloco] = exportar_previsoes(
                nome_bloco, datas, y, np.array([]), paths
            )
            continue
        y_pred = model.predict(scaler.transform(X), verbose=0).reshape(-1)
        metricas_por_bloco[nome_bloco] = exportar_previsoes(
            nome_bloco, datas, y, y_pred, paths
        )
        previsoes_por_bloco[nome_bloco] = pd.DataFrame(
            {"datahora": pd.to_datetime(datas), "y_real": y, "y_pred": y_pred}
        )

    salvar_json(metricas_por_bloco, paths["metricas"] / "metricas_todos_blocos.json")
    return {
        "experimento": nome_experimento,
        "paths": paths,
        "metricas": metricas_por_bloco,
        "previsoes": previsoes_por_bloco,
        "input_dim": int(X_train_scaled.shape[1]),
        "n_treino": int(len(y_train)),
        "n_validacao": int(len(y_val)),
    }


def extrair_metrica_resultado(
    resultado: dict[str, Any], bloco_preferido: str, metrica: str
) -> float:
    metricas_bloco = resultado["metricas"].get(bloco_preferido, {})
    valor = metricas_bloco.get(metrica, np.nan)
    if pd.isna(valor):
        valor = resultado["metricas"].get("teste", {}).get(metrica, np.nan)
    return float(valor)


def salvar_comparativo_janelas(
    resultados_janelas: list[dict[str, Any]], config: dict[str, Any], output_dir: Path
) -> pd.DataFrame:
    linhas = []
    bloco = str(config["comparar_por_bloco"])
    for resultado in resultados_janelas:
        janela = int(resultado["experimento"].split("_")[-1])
        for nome_bloco, metricas in resultado["metricas"].items():
            linhas.append({"janela": janela, "bloco": nome_bloco, **metricas})
    df_metricas = pd.DataFrame(linhas)
    df_metricas.to_csv(output_dir / "comparativo_janelas_metricas.csv", index=False)

    df_plot = df_metricas[df_metricas["bloco"] == bloco].copy()
    if df_plot.empty:
        df_plot = df_metricas[df_metricas["bloco"] == "teste"].copy()
    if not df_plot.empty:
        for metrica in ["mae", "mse", "rmse", "erro_minimo", "erro_maximo", "r2", "pearson_r"]:
            plt.figure(figsize=(10, 5))
            plt.plot(df_plot["janela"], df_plot[metrica], marker="o")
            plt.title(
                f"Janelamento 1 até N - {metrica.upper()} ({df_plot['bloco'].iloc[0]})"
            )
            plt.xlabel("Tamanho da janela")
            plt.ylabel(metrica.upper())
            plt.tight_layout()
            plt.savefig(output_dir / f"comparativo_janelas_{metrica}.png", dpi=300)
            plt.close()
    return df_metricas


def salvar_comparativo_tecnicas(
    resultados: list[dict[str, Any]],
    melhor_janelamento: dict[str, Any],
    config: dict[str, Any],
    output_dir: Path,
) -> pd.DataFrame:
    """
    Gera comparativo consolidado entre as três técnicas principais:
    - Discretização simples
    - Discretização trigonométrica
    - Melhor janelamento

    Agora o comparativo é feito para todos os blocos:
    treino, validacao, teste e previsao.
    """
    selecionados = [
        next(r for r in resultados if r["experimento"] == "discretizacao_simples"),
        next(r for r in resultados if r["experimento"] == "discretizacao_trigonometrica"),
        melhor_janelamento,
    ]

    nomes = {
        "discretizacao_simples": "Discretização simples",
        "discretizacao_trigonometrica": "Discretização trigonométrica",
        melhor_janelamento["experimento"]: (
            f"Melhor janelamento (n = {melhor_janelamento['experimento'].split('_')[-1]})"
        ),
    }

    # Uma linha para cada combinação técnica x bloco.
    linhas = []
    for resultado in selecionados:
        for nome_bloco, metricas in resultado["metricas"].items():
            linhas.append(
                {
                    "tecnica": nomes[resultado["experimento"]],
                    "experimento": resultado["experimento"],
                    "bloco": nome_bloco,
                    **metricas,
                }
            )

    df_comp = pd.DataFrame(linhas)
    
    colunas_ordenadas = [
        "tecnica",
        "experimento",
        "bloco",
        "mae",
        "mse",
        "rmse",
        "erro_minimo",
        "erro_maximo",
        "pearson_r",
        "r2",
        "n_amostras",
    ]
    colunas_existentes = [col for col in colunas_ordenadas if col in df_comp.columns]
    df_comp = df_comp[colunas_existentes]

    # Arquivo principal desejado: todas as técnicas x todos os blocos.
    df_comp.to_csv(output_dir / "comparativo_tecnicas_metricas_todos_blocos.csv", index=False)

    # Mantém também um nome curto para compatibilidade com o restante do pipeline.
    df_comp.to_csv(output_dir / "comparativo_tecnicas_metricas.csv", index=False)

    metricas_plot = ["mae", "mse", "erro_minimo", "erro_maximo", "pearson_r", "r2"]

    # Gráficos comparativos por bloco.
    for nome_bloco in df_comp["bloco"].dropna().unique():
        df_bloco = df_comp[df_comp["bloco"] == nome_bloco].copy()

        for metrica in metricas_plot:
            if metrica not in df_bloco.columns:
                continue

            plt.figure(figsize=(11, 5))
            plt.bar(df_bloco["tecnica"], df_bloco[metrica])
            plt.title(f"Comparativo de técnicas - {metrica.upper()} ({nome_bloco})")
            plt.xlabel("Técnica")
            plt.ylabel(metrica.upper())
            plt.xticks(rotation=15, ha="right")
            plt.tight_layout()
            plt.savefig(
                output_dir / f"comparativo_tecnicas_{nome_bloco}_{metrica}.png",
                dpi=300,
            )
            plt.close()

    # Série real vs previstas no mesmo gráfico para cada bloco.
    for nome_bloco in ["treino", "validacao", "teste", "previsao"]:
        plt.figure(figsize=(14, 5))
        primeiro = True
        tem_dados = False

        for resultado in selecionados:
            prev = resultado["previsoes"].get(nome_bloco)
            if prev is None or prev.empty:
                continue

            tem_dados = True

            if primeiro:
                plt.plot(prev["datahora"], prev["y_real"], label="Real", linestyle="--")
                primeiro = False

            plt.plot(
                prev["datahora"],
                prev["y_pred"],
                label=nomes[resultado["experimento"]],
            )

        if not tem_dados:
            plt.close()
            continue

        plt.title(f"Comparativo das previsões por técnica - {nome_bloco}")
        plt.xlabel("Data e hora")
        plt.ylabel("v_anemo2 (m/s)")
        plt.legend()
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y %H:%M"))
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(output_dir / f"comparativo_tecnicas_series_{nome_bloco}.png", dpi=300)
        plt.close()

    return df_comp

In [ ]:
def executar_pipeline(config: dict[str, Any]) -> None:
    configurar_semente(int(config["random_seed"]))
    df = carregar_dataset(str(config["csv_path"]))
    validar_dataset(df, str(config["target_col"]))

    output_dir = Path(str(config["output_dir"]))
    output_dir.mkdir(parents=True, exist_ok=True)

    print(40*"==")
    print("Dataset carregado:")
    print(f"- Registros: {len(df)}")
    print(f"- Período: {df['datahora'].min()} até {df['datahora'].max()}")
    print(f"- Anos disponíveis: {sorted(df['ano'].dropna().astype(int).unique().tolist())}")
    print(40*"==")
    
    datasets = {
        "discretizacao_simples": criar_dataset_discretizacao_simples(df, str(config["target_col"])),
        "discretizacao_trigonometrica": criar_dataset_discretizacao_trigonometrica(df, str(config["target_col"])),
    }
    for janela in range(1, int(config["max_janela"]) + 1):
        datasets[f"janelamento_{janela}"] = criar_dataset_janelamento(df, str(config["target_col"]), janela)

    resultados: list[dict[str, Any]] = []
    for nome_experimento, df_modelo in datasets.items():
        print(f"\nTreinando: {nome_experimento} | amostras: {len(df_modelo)} | features: {len(selecionar_features(df_modelo))}")
        resultados.append(treinar_e_avaliar(df_modelo, config, nome_experimento))
        print()
        print(20*"####")

    resultados_janelas = [r for r in resultados if r["experimento"].startswith("janelamento_")]
    salvar_comparativo_janelas(resultados_janelas, config, output_dir)

    metrica = str(config["comparar_por_metrica"])
    bloco = str(config["comparar_por_bloco"])
    melhor_janelamento = min(resultados_janelas, key=lambda r: extrair_metrica_resultado(r, bloco, metrica))
    df_comp = salvar_comparativo_tecnicas(resultados, melhor_janelamento, config, output_dir)

    resumo = {
        "melhor_janelamento": melhor_janelamento["experimento"],
        "criterio": {"bloco": bloco, "metrica": metrica, "menor_valor_eh_melhor": True},
        "comparativo_tecnicas": df_comp.to_dict(orient="records"),
    }
    salvar_json(resumo, output_dir / "resumo_final.json")

    print("\nResumo comparativo - todas as técnicas e todos os blocos:")
    colunas_resumo = [
        "tecnica",
        "bloco",
        "mae",
        "mse",
        "rmse",
        "erro_minimo",
        "erro_maximo",
        "pearson_r",
        "r2",
        "n_amostras",
    ]
    display(df_comp[colunas_resumo].rename(columns={"pearson_r": "r", "r2": "r2"}).sort_values(by='bloco'))

    print(f"\nMelhor janelamento: {melhor_janelamento['experimento']}")
    print(f"\nResultados salvos em: {output_dir.resolve()}")

In [ ]:
executar_pipeline(CONFIG)